# Lab 28 — Kaggle GPU Serving (Complete)

Notebook này khởi động vLLM, embedding service 384 chiều, một public ngrok tunnel dùng chung, và ghi deployment metadata vào MLflow.

Trước khi chạy: bật **GPU Accelerator** và **Internet** trong Kaggle Settings. Tạo Kaggle Secret `NGROK_AUTHTOKEN`. Các secret MLflow là tùy chọn: `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`.

In [ ]:
%pip install -q "vllm==0.10.2" --extra-index-url https://download.pytorch.org/whl/cu128
%pip install -q "fastapi==0.115.12" "starlette==0.46.2" uvicorn pyngrok "sentence-transformers==3.4.1" "mlflow>=2.10,<3" httpx requests "protobuf<6"
%pip uninstall -y -q torchcodec


## Bắt buộc: restart sau khi cài đặt

Sau khi cell cài đặt hoàn tất, chọn **Run → Restart Session**. Sau khi kernel kết nối lại, bắt đầu chạy từ cell **Validation sau restart** ngay bên dưới; không chạy lại cell cài đặt. Các cảnh báo conflict với BigFrames, Google ADK, Gradio, RAPIDS hoặc `datasets` không ảnh hưởng notebook vì các package đó không được sử dụng.

## Validation sau restart

In [ ]:
import subprocess, sys
import torch
import vllm
import fastapi
import starlette
import sentence_transformers
import mlflow

versions = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "vllm": vllm.__version__,
    "fastapi": fastapi.__version__,
    "starlette": starlette.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "mlflow": mlflow.__version__,
}
print(versions)
assert torch.cuda.is_available(), "GPU chưa bật: mở Kaggle Settings → Accelerator → GPU"
assert tuple(map(int, vllm.__version__.split(".")[:2])) == (0, 10), f"Sai vLLM version: {vllm.__version__}"
gpu_name = torch.cuda.get_device_name(0)
compute_capability = torch.cuda.get_device_capability(0)
print("GPU:", gpu_name, "compute capability:", compute_capability)
assert compute_capability >= (7, 0), (
    f"GPU {gpu_name} có compute capability {compute_capability}, nhưng vLLM cần >= (7, 0). "
    "Hãy đổi Kaggle Accelerator từ P100 sang T4 x2, restart session và chạy lại."
)
print("VALIDATION OK — có thể tiếp tục chạy các cell bên dưới")


## 1. Cấu hình và kiểm tra GPU

In [ ]:
import os, subprocess, time, threading, json, requests
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4")
VLLM_PORT = 8001
EDGE_PORT = 8000
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
secrets = UserSecretsClient()

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True, check=True)
print(gpu.stdout)
GPU_COUNT = len([line for line in gpu.stdout.splitlines() if line.strip()])
assert GPU_COUNT >= 1, "Kaggle GPU is not enabled"
print({"model": MODEL_NAME, "gpu_count": GPU_COUNT})


## 2. Khởi động vLLM OpenAI-compatible server

In [ ]:
vllm_log = open("/kaggle/working/vllm.log", "w")
cmd = [
    "vllm", "serve", MODEL_NAME,
    "--host", "0.0.0.0",
    "--port", str(VLLM_PORT),
    "--dtype", "half",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.75",
    "--trust-remote-code",
]
# Chỉ bật tensor parallel khi notebook có nhiều GPU.
if GPU_COUNT > 1:
    cmd += ["--tensor-parallel-size", str(GPU_COUNT)]

vllm_process = subprocess.Popen(cmd, stdout=vllm_log, stderr=subprocess.STDOUT)
print("vLLM PID:", vllm_process.pid)

def wait_for_url(url, timeout=600):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if vllm_process.poll() is not None:
            print(Path("/kaggle/working/vllm.log").read_text()[-5000:])
            raise RuntimeError("vLLM exited before becoming healthy")
        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                return response
        except requests.RequestException:
            pass
        time.sleep(5)
    raise TimeoutError(f"Timed out waiting for {url}")

wait_for_url(f"http://127.0.0.1:{VLLM_PORT}/health")
print("vLLM is healthy")


## 3. Embedding service và edge gateway dùng chung một tunnel

In [ ]:
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
import httpx, uvicorn

embedding_model = SentenceTransformer(EMBED_MODEL, device="cpu")
edge_app = FastAPI(title="Lab28 Kaggle GPU Edge")

class EmbedRequest(BaseModel):
    texts: list[str] = Field(min_length=1, max_length=64)

@edge_app.post("/embed")
def embed(body: EmbedRequest):
    vectors = embedding_model.encode(body.texts, normalize_embeddings=True).tolist()
    return {"embeddings": vectors, "dimension": len(vectors[0]), "model": EMBED_MODEL}

@edge_app.get("/health")
async def health():
    async with httpx.AsyncClient(timeout=5) as client:
        response = await client.get(f"http://127.0.0.1:{VLLM_PORT}/health")
    return {"status": "ok", "vllm_status": response.status_code}

@edge_app.api_route("/v1/{path:path}", methods=["GET", "POST"])
async def proxy_vllm(path: str, request: Request):
    body = await request.body()
    headers = {"content-type": request.headers.get("content-type", "application/json")}
    async with httpx.AsyncClient(timeout=120) as client:
        response = await client.request(request.method, f"http://127.0.0.1:{VLLM_PORT}/v1/{path}", content=body, headers=headers)
    return __import__("fastapi").Response(content=response.content, status_code=response.status_code, media_type=response.headers.get("content-type"))

edge_config = uvicorn.Config(edge_app, host="0.0.0.0", port=EDGE_PORT, log_level="info")
edge_server = uvicorn.Server(edge_config)
threading.Thread(target=edge_server.run, daemon=True).start()

deadline = time.time() + 120
while time.time() < deadline:
    try:
        if requests.get(f"http://127.0.0.1:{EDGE_PORT}/health", timeout=5).status_code == 200:
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Edge gateway did not start")
print("Embedding + edge gateway are healthy")


## 4. Mở ngrok tunnel

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(secrets.get_secret("NGROK_AUTHTOKEN"))
ngrok.kill()  # dọn tunnel cũ trong kernel này
public_tunnel = ngrok.connect(EDGE_PORT, "http")
PUBLIC_URL = public_tunnel.public_url.replace("http://", "https://")
print("VLLM_NGROK_URL=" + PUBLIC_URL)
print("EMBED_NGROK_URL=" + PUBLIC_URL)
print("Giữ notebook/kernel này chạy để tunnel không bị đóng.")


## 5. Kiểm tra embedding và chat qua public URL

In [ ]:
embed_response = requests.post(PUBLIC_URL + "/embed", json={"texts": ["AI platform integration test"]}, timeout=60)
embed_response.raise_for_status()
assert embed_response.json()["dimension"] == 384
print("Embedding OK:", embed_response.json()["dimension"], "dimensions")

chat_response = requests.post(
    PUBLIC_URL + "/v1/chat/completions",
    json={"model": MODEL_NAME, "messages": [{"role": "user", "content": "Reply with: Lab28 vLLM OK"}], "max_tokens": 32},
    timeout=120,
)
chat_response.raise_for_status()
print(chat_response.json()["choices"][0]["message"]["content"])


## 6. Ghi deployment metadata vào MLflow

In [ ]:
import mlflow

def optional_secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return None

tracking_uri = optional_secret("MLFLOW_TRACKING_URI") or "file:///kaggle/working/mlruns"
tracking_username = optional_secret("MLFLOW_TRACKING_USERNAME")
tracking_password = optional_secret("MLFLOW_TRACKING_PASSWORD")
if tracking_username:
    os.environ["MLFLOW_TRACKING_USERNAME"] = tracking_username
if tracking_password:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = tracking_password

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("lab28-integration")
with mlflow.start_run(run_name="vllm-serving-kaggle") as run:
    mlflow.log_params({"model": MODEL_NAME, "embedding_model": EMBED_MODEL, "gpu_count": GPU_COUNT, "vector_dimension": 384})
    mlflow.set_tags({"serving_url": PUBLIC_URL, "deployment_status": "production", "registry_alias": "champion"})
    RUN_ID = run.info.run_id
print({"tracking_uri": tracking_uri, "run_id": RUN_ID})


## 7. Giá trị cần copy về `.env` local

In [ ]:
print(f"""# Copy hai dòng này vào .env local
VLLM_NGROK_URL={PUBLIC_URL}
EMBED_NGROK_URL={PUBLIC_URL}

# Sau đó chạy local:
# docker compose up -d --force-recreate api-gateway
# python scripts/05_embed_to_qdrant.py
# python scripts/06_register_vllm_mlflow.py
# pytest smoke-tests -v
""")


## Troubleshooting

- Nếu vLLM hết VRAM: giảm `--gpu-memory-utilization` xuống `0.65`, đổi `--max-model-len` thành `2048`, hoặc dùng `Qwen/Qwen2.5-3B-Instruct`.
- Nếu model cần quyền truy cập: thêm Kaggle Secret `HF_TOKEN` và đặt `os.environ['HF_TOKEN']` trước cell vLLM.
- Nếu ngrok báo giới hạn session: đóng notebook/tunnel cũ tại ngrok dashboard rồi chạy lại cell tunnel.
- URL chỉ tồn tại khi Kaggle session còn chạy. Không commit token hoặc URL có thông tin xác thực vào Git.